In [1]:
import json
import math
import random

import pandas as pd

from pathlib import Path
from collections import defaultdict

In [2]:
PROJECT_DIR = Path.cwd()

# ANALYSIS_DIR = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\analysis\\all_schemes.json"
OUTPUT_DIR = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset"

CATEGORY_FILE = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\output\\category_scheme_ids.json"
SUMMARY_FILE = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\output\\category_summary.csv"

RANDOM_SEED = 42
SAMPLE_PERCENTAGE = 0.10

random.seed(RANDOM_SEED)

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [3]:
with open(CATEGORY_FILE, "r", encoding="utf-8") as file:
    category_scheme_ids = json.load(file)

category_df = pd.read_csv(SUMMARY_FILE)

print("Total categories:", len(category_scheme_ids))

category_df.head()

Total categories: 15


,category,total_schemes,sample_10_percent
0,Social welfare & Empowerment,1331,134
1,Education & Learning,1023,103
2,"Agriculture,Rural & Environment",808,81
3,Business & Entrepreneurship,730,73
4,Women and Child,426,43


In [4]:
# Step 4 — Calculate sampling quota
# We will use math.ceil() so that every category gets at least its rounded-up 10% quota

category_quotas = {}

for category, scheme_ids in category_scheme_ids.items():

    total_schemes = len(scheme_ids)

    sample_size = math.ceil(
        total_schemes * SAMPLE_PERCENTAGE
    )

    category_quotas[category] = {
        "total_schemes": total_schemes,
        "sample_size": sample_size
    }

quota_df = pd.DataFrame([
    {
        "category": category,
        "total_schemes": values["total_schemes"],
        "sample_size": values["sample_size"]
    }
    for category, values in category_quotas.items()
])

quota_df = quota_df.sort_values(
    "total_schemes",
    ascending=False
).reset_index(drop=True)

quota_df

,category,total_schemes,sample_size
0,Social welfare & Empowerment,1331,134
1,Education & Learning,1023,103
2,"Agriculture,Rural & Environment",808,81
3,Business & Entrepreneurship,730,73
4,Women and Child,426,43
5,Skills & Employment,368,37
6,"Banking,Financial Services and Insurance",308,31
7,Health & Wellness,263,27
8,Sports & Culture,261,27
9,Housing & Shelter,126,13


In [5]:
# Step 5 — Randomly select scheme IDs category-wise
# This is the main sampling operation.
# 
# A scheme can be selected through multiple categories, so we will preserve all the categories through which it was selected.

selected_scheme_ids = set()

selected_for_categories = defaultdict(list)

category_sampling_results = {}

for category, scheme_ids in category_scheme_ids.items():

    quota = category_quotas[category]["sample_size"]

    # Random selection within each category
    sampled_ids = random.sample(
        scheme_ids,
        k=min(quota, len(scheme_ids))
    )

    category_sampling_results[category] = sampled_ids

    for scheme_id in sampled_ids:

        selected_scheme_ids.add(scheme_id)

        selected_for_categories[scheme_id].append(category)


print("Category sampling completed!")

print("Unique selected schemes:", len(selected_scheme_ids))
print(
    "Total category-wise selections:",
    sum(len(ids) for ids in category_sampling_results.values())
)

Category sampling completed!
Unique selected schemes: 592
Total category-wise selections: 612


In [6]:
# Step 6 — Analyze category coverage after deduplication
# We need to distinguish the requested category quota from the number of unique schemes selected for that category.

coverage_results = []

for category, scheme_ids in category_scheme_ids.items():

    requested = category_quotas[category]["sample_size"]

    selected_count = len(
        set(category_sampling_results[category])
    )

    coverage_results.append({

        "category": category,

        "total_schemes": len(scheme_ids),

        "target_sample": requested,

        "selected_in_category": selected_count,

        "coverage_percentage": round(
            (selected_count / len(scheme_ids)) * 100,
            2
        )
    })


coverage_df = pd.DataFrame(coverage_results)

coverage_df = coverage_df.sort_values(
    "total_schemes",
    ascending=False
).reset_index(drop=True)

coverage_df

,category,total_schemes,target_sample,selected_in_category,coverage_percentage
0,Social welfare & Empowerment,1331,134,134,10.07
1,Education & Learning,1023,103,103,10.07
2,"Agriculture,Rural & Environment",808,81,81,10.02
3,Business & Entrepreneurship,730,73,73,10.00
4,Women and Child,426,43,43,10.09
5,Skills & Employment,368,37,37,10.05
6,"Banking,Financial Services and Insurance",308,31,31,10.06
7,Health & Wellness,263,27,27,10.27
8,Sports & Culture,261,27,27,10.34
9,Housing & Shelter,126,13,13,10.32


In [ ]:
# Important: Deduplication does not reduce the count selected within an individual category. 
# If a scheme is selected in three categories, it still counts once in each of those category samples, 
# but only once in the overall unique scheme collection.

In [7]:
# Step 7 — Create scheme-level selection mapping
# This preserves the original categories and the categories through which each scheme was selected.

scheme_selection_mapping = {}

# Create lookup of scheme metadata
scheme_lookup = {}

for category, scheme_ids in category_scheme_ids.items():

    for scheme_id in scheme_ids:

        if scheme_id not in scheme_lookup:
            scheme_lookup[scheme_id] = []

        scheme_lookup[scheme_id].append(category)


for scheme_id in sorted(selected_scheme_ids):

    scheme_selection_mapping[scheme_id] = {

        "scheme_id": scheme_id,

        "categories": scheme_lookup[scheme_id],

        "selected_for": selected_for_categories[scheme_id]

    }


print("Selection mapping created!")

print("Total unique schemes:", len(scheme_selection_mapping))

Selection mapping created!
Total unique schemes: 592


In [9]:
# Step 8 — Save sampling results
# Save unique selected scheme IDs
selected_ids_file = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\selected_scheme_ids.json"

with open(selected_ids_file, "w", encoding="utf-8") as file:

    json.dump(
        sorted(selected_scheme_ids),
        file,
        ensure_ascii=False,
        indent=4
    )


# Save detailed selection mapping
mapping_file = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\scheme_selection_mapping.json"

with open(mapping_file, "w", encoding="utf-8") as file:

    json.dump(
        scheme_selection_mapping,
        file,
        ensure_ascii=False,
        indent=4
    )


# Save category coverage
coverage_file = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\sampling_coverage.csv"

coverage_df.to_csv(
    coverage_file,
    index=False
)


print("Sampling results saved!")

Sampling results saved!


In [10]:
# Step 9 — Fetch original scheme JSON records
# Now we need to fetch the full scheme information using the selected IDs.

# For this step, load your original JSON file containing all schemes.
SOURCE_FILE = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\analysis\\all_schemes.json"

with open(SOURCE_FILE, "r", encoding="utf-8") as file:
    all_schemes = json.load(file)

if isinstance(all_schemes, dict):
    all_schemes = [all_schemes]


# Create scheme ID lookup
all_scheme_lookup = {}

for scheme in all_schemes:

    scheme_id = str(
        scheme.get("scheme_id", "")
    ).strip()

    if scheme_id:

        if scheme_id in all_scheme_lookup:
            raise ValueError(
                f"Duplicate scheme ID found in source JSON: {scheme_id}"
            )

        all_scheme_lookup[scheme_id] = scheme


# Fetch complete records
selected_schemes = []

missing_scheme_ids = []

for scheme_id in sorted(selected_scheme_ids):

    scheme = all_scheme_lookup.get(scheme_id)

    if scheme is None:

        missing_scheme_ids.append(scheme_id)
        continue

    selected_schemes.append(scheme)


print("Selected scheme records:", len(selected_schemes))
print("Missing scheme IDs:", len(missing_scheme_ids))

Selected scheme records: 592
Missing scheme IDs: 0


In [11]:
# Step 10 — Save complete selected scheme records

selected_schemes_file = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\selected_schemes.json"

with open(
    selected_schemes_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        selected_schemes,
        file,
        ensure_ascii=False,
        indent=4
    )


print("Complete selected scheme dataset saved!")
print("File:", selected_schemes_file)

Complete selected scheme dataset saved!
File: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\selected_schemes.json


In [13]:
print("========== SEVA AI SAMPLING REPORT ==========")

print("Total original schemes:", len(all_schemes))

print("Total categories:", len(category_scheme_ids))

print("Unique selected scheme IDs:", len(selected_scheme_ids))

print("Total fetched scheme records for golden datasets:", len(selected_schemes))

print("Missing scheme IDs:", len(missing_scheme_ids))

print(
    "Total original FAQ records:",
    sum(
        len(scheme.get("faqs", []))
        for scheme in selected_schemes
    )
)

print("=============================================")

========== SEVA AI SAMPLING REPORT ==========
Total original schemes: 4428
Total categories: 15
Unique selected scheme IDs: 592
Total fetched scheme records for golden datasets: 592
Missing scheme IDs: 0
Total original FAQ records: 6485
